In [ ]:
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch_geometric.utils import softmax
import math

In [ ]:

def get_graph(raw_data, sensor_num = 6):
    sensor_data = torch.tensor(raw_data, dtype=torch.float)

    n = sensor_data.shape[0] # sequence_length
    node_features = sensor_data.contiguous().view(-1, 3) 

    edge_index = []
    edge_type = []


    for t in range(n-1): 
        for sensor_id in range(sensor_num):  # 0: Acc, 1: Err_Acc 2: Gyro, 3: Err_Gyro 4: Mag 5: Err_Mag
            edge_index.append([sensor_num * t + sensor_id, sensor_num * (t+1) + sensor_id])  
            edge_type.append(0)  


    for t in range(n):
        edge_index.append([sensor_num * t, sensor_num * t + 2])  # Acc -> Gyro
        edge_index.append([sensor_num * t + 4, sensor_num * t + 2])  # Mag -> Gyro
        edge_index.append([sensor_num * t + 2, sensor_num * t])  # Gyro -> Acc
        edge_index.append([sensor_num * t + 2, sensor_num * t + 4])  # Gyro -> Mag
        edge_type.extend([1, 2, 3, 4])  


    for t in range(n):
        edge_index.append([sensor_num * t, sensor_num * t + 1])  # Acc -> Error Acc
        edge_index.append([sensor_num * t + 2, sensor_num * t + 3])  # Gyro -> Error Gyro
        edge_index.append([sensor_num * t + 4, sensor_num * t + 5])  # Mag -> Error Mag
        edge_type.extend([5, 5, 5])  

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_type = torch.tensor(edge_type, dtype=torch.long) 

    return node_features, edge_index, edge_type

def data_to_model(data, sequence_length=120, step_size=25):
    data = torch.tensor(data, dtype=torch.float32)
    sequences = []
    N = data.shape[0]
    
    for start_idx in range(0, N - sequence_length + 1, step_size):
        sequence = data[start_idx:start_idx + sequence_length]
        sequences.append(sequence)
    
    sequences = torch.stack(sequences)
    
    return sequences

def get_data(mode = '1-hover', phase = '2_5'):
    # 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
    fmode = mode
    base_path = "..."

    sycn_raw_data = f'all_sycn_raw_data_mode{phase}.csv'
    sycn_err_data = f'all_sycn_err_data_mode{phase}.csv'
    sycn_sim_data = f'all_sycn_sim_data_mode{phase}.csv'

    sycn_raw_data_path = os.path.join(base_path, 'data_for_r2v_c4', 'Dpro', fmode, sycn_raw_data)
    sycn_err_data_path = os.path.join(base_path, 'data_for_r2v_c4', 'Dpro', fmode, sycn_err_data)
    sycn_sim_data_path = os.path.join(base_path, 'data_for_r2v_c4', 'Dpro', fmode, sycn_sim_data)

    raw_data = pd.read_csv(sycn_raw_data_path).to_numpy()
    err_data = pd.read_csv(sycn_err_data_path).to_numpy()
    sim_data = pd.read_csv(sycn_sim_data_path).to_numpy()
    return raw_data, sim_data, err_data

def get_train_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3']):
    fi_raw, fi_sim, fi_err = get_data(mode= mode[0], phase=phase[0])
    sim_data = np.vstack((fi_sim))
    raw_data = np.vstack((fi_raw))
    err_data = np.vstack((fi_err))
    for mo, ph in zip(mode[1:], phase[1:]):
        raw, sim, err = get_data(mode= mo, phase= ph)
        sim_data = np.vstack((sim_data, sim))
        raw_data = np.vstack((raw_data, raw))
        err_data = np.vstack((err_data, err))

    sim_data_split = np.split(sim_data, indices_or_sections=3, axis=1)
    raw_data_split = np.split(raw_data, indices_or_sections=3, axis=1)
    err_data_split = np.split(err_data, indices_or_sections=3, axis=1)

    sim_gyro, sim_acc, sim_mag = sim_data_split[0], sim_data_split[1], sim_data_split[2]    
    raw_gyro, raw_acc, raw_mag = raw_data_split[0], raw_data_split[1], raw_data_split[2]
    err_gyro, err_acc, err_mag = err_data_split[0], err_data_split[1], err_data_split[2]

    sim_data_ = np.hstack((sim_acc, sim_gyro, sim_mag))
    raw_data_ = np.hstack((raw_acc, raw_gyro, raw_mag))
    err_data_ = np.hstack((err_acc, err_gyro, err_mag))

    return sim_data_, raw_data_, err_data_

def get_sensor_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3'], label = 1, shuffle=False, ratio=1, sequence_length=80, step_size=1):
    sim, raw, err = get_train_data(mode, phase)
    raw_acc, raw_gyro, raw_mag = raw[:,0:3], raw[:,3:6], raw[:,6:9]
    err_acc, err_gyro, err_mag = err[:,0:3], err[:,3:6], err[:,6:9]
    raw_err = np.hstack((raw_acc, err_acc, raw_gyro, err_gyro, raw_mag, err_mag))

    Q_acc_raw = np.std(raw_acc, axis=0) / abs(np.mean(raw_acc, axis=0))
    Q_gyro_raw = np.std(raw_gyro, axis=0) / abs(np.mean(raw_gyro, axis=0))
    Q_mag_raw = np.std(raw_mag, axis=0) / abs(np.mean(raw_mag, axis=0))

    Q_acc_raw, Q_gyro_raw, Q_mag_raw = np.mean(Q_acc_raw), np.mean(Q_gyro_raw), np.mean(Q_mag_raw)
    max_Q = max(Q_acc_raw, Q_gyro_raw, Q_mag_raw)

    gama = 1
    omega_q = {
        'a2g': max_Q / ((Q_acc_raw + Q_gyro_raw)**gama),
        'm2g': max_Q / ((Q_mag_raw + Q_gyro_raw)**gama),
        'g2a': max_Q / ((Q_gyro_raw + Q_acc_raw)**gama),
        'g2m': max_Q / ((Q_gyro_raw + Q_mag_raw)**gama),
    }
    
    tao = 10**1
    C_ij = {
        'a2g': tao * np.mean(np.mean(np.cov(raw_acc, raw_gyro, rowvar=False)[:3, 3:], axis=1)),
        'm2g': tao * np.mean(np.mean(np.cov(raw_mag, raw_gyro, rowvar=False)[:3, 3:], axis=1)),
        'g2a': tao * np.mean(np.mean(np.cov(raw_acc, raw_gyro, rowvar=False)[:3, 3:], axis=1)),
        'g2m': tao * np.mean(np.mean(np.cov(raw_mag, raw_gyro, rowvar=False)[:3, 3:], axis=1)),
    }

    Epsilon_ij = {
        'a2g': omega_q['a2g'] * math.exp(-C_ij['a2g']),
        'm2g': omega_q['m2g'] * math.exp(-C_ij['m2g']),
        'g2a': omega_q['g2a'] * math.exp(-C_ij['g2a']),
        'g2m': omega_q['g2m'] * math.exp(-C_ij['g2m'])
    }

    raw_err_2_model = data_to_model(data=raw_err, sequence_length=sequence_length, step_size=step_size)
    raw_err_2_model_labels = torch.full((raw_err_2_model.size(0), 1), label, dtype=torch.long)

    if shuffle:
        torch.manual_seed(42)
        indices = torch.randperm(raw_err_2_model.size(0))
        raw_err_2_model = raw_err_2_model[indices]
        raw_err_2_model_labels = raw_err_2_model_labels[indices]
    raw_err_2_model = raw_err_2_model[:int(raw_err_2_model.size(0) * ratio)]
    raw_err_2_model_labels = raw_err_2_model_labels[:int(raw_err_2_model_labels.size(0) * ratio)]

    return raw_err_2_model, raw_err_2_model_labels, Epsilon_ij

In [ ]:

'''Step 1: Get [raw] and [err] data'''
# 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
normal_2_model, normal_lables, Epsilon_ij = get_sensor_data(mode= ['1-hover-normal-r2vo','1-hover-normal-r2vo', '2-forward-normal-r2vo'], phase = ['2_3', '2_6', '2_5'], label = 1, shuffle=True, ratio=1, sequence_length=80, step_size=4)
af3_2_model, af3_labels, _ =  get_sensor_data(mode= ['3-acc-x-axis-flag3-3000-normal-r2vo'], phase = ['2_9'], label = 2, shuffle=True, ratio=1, sequence_length=80, step_size=4)
mf3_2_model, mf3_labels, _ =  get_sensor_data(mode= ['6-motor03-flag3-200-normal-r2vo'], phase = ['2_9'], label = 3, shuffle=True, ratio=1, sequence_length=80, step_size=4)
mf4_2_model, mf4_labels, _ =  get_sensor_data(mode= ['5-motor03-flag4-085-normal-r2vo'], phase = ['2_9'], label = 4, shuffle=True, ratio=1, sequence_length=80, step_size=4)

print(f'normal_2_model:{normal_2_model.shape} af3_2_model:{af3_2_model.shape} mf3_2_model:{mf3_2_model.shape}  mf4_2_model:{mf4_2_model.shape}')
print(f'normal_lables:{normal_lables.shape} af3_labels:{af3_labels.shape} mf3_labels:{mf3_labels.shape}  mf4_labels:{mf4_labels.shape}')

data_2_model = torch.cat((normal_2_model, af3_2_model, mf3_2_model, mf4_2_model), dim=0)
labels_2_model = torch.cat((normal_lables, af3_labels, mf3_labels, mf4_labels), dim=0).squeeze(1)

num_classes = 4
one_hot = torch.zeros(labels_2_model.size(0), num_classes, dtype=torch.float)
for i in range(labels_2_model.size(0)):
    one_hot[i][labels_2_model[i] - 1] = 1
label_2_model = one_hot
print(f'data_2_model:{data_2_model.shape} label_2_model:{label_2_model.shape}')


In [ ]:
class MyGATConv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, concat=True, negative_slope=0.2, dropout=0.3, gamma=1.0, **kwargs):
        super(MyGATConv, self).__init__(aggr='add', **kwargs)
        '''
        in_channels, out_channels: dimensions of input and output features.
        heads: number of heads of the attention mechanism, indicating how many independent attention mechanisms are used in the multi-head attention mechanism.
        concat: whether to concatenate the outputs of multiple heads together.
        negative_slope: negative slope parameter of the LeakyReLU activation function.
        dropout: used to randomly drop part of the attention weights to prevent overfitting.
        gamma: weight of the penalty term, used to control the effect of the distance penalty on the attention weights.
        self.weight: stores the weight of the linear transformation from input features to output features.
        self.att: stores the parameters used to calculate the attention weights.
        '''
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.negative_slope = negative_slope
        self.dropout = dropout
        self.gamma = gamma

        self.weight = nn.Parameter(torch.Tensor(heads, in_channels, out_channels))
        self.att = nn.Parameter(torch.Tensor(heads, 2 * out_channels))

        self.leaky_relu = nn.LeakyReLU(self.negative_slope)
        self.attention_dropout = nn.Dropout(p=dropout)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.att)

    def forward(self, x, edge_index, edge_type):
        # Step 1: Linearly transform node feature matrix.
        '''
        Shape:
        x: [N, in_channels]  (Physical Meaning: Each node(N) has in_channels's features (3-axis sensoor data))
        x.unsqueeze(1): [N, 1, in_channels]
        self.weight: [num_heads, in_channels, out_channels]
        x('nid,hdo->nho'): [N, num_heads, out_channels] (Physical Meaning: Each head(num_heads) of each node(N) outputs features(out_channels))
        '''
        x = torch.einsum('nid,hdo->nho', x.unsqueeze(1), self.weight)

        # Step 2: Start propagating messages.
        # 2.1）: Compute attention coefficients using custom logic.
        '''
        Shape:
        x_i: [E, num_heads, out_channels] (Source node features)
        x_j: [E, num_heads, out_channels] (Target node features)
        x_concat: [E, num_heads, 2 * out_channels]
        self.att: [E, 2 * out_channels]
        alpha: [E, num_heads]
        distance: [E, num_heads]

        att^T [Wh_i || Wh_j] --> [E, num_heads, 2 * out_channels] --> sum(dim=-1) --> [E, num_heads]
        '''
        row, col = edge_index
        x_i = x[row]
        x_j = x[col]
        x_concat = torch.cat([x_i, x_j], dim=-1)
        alpha = (x_concat * self.att).sum(dim=-1) # alpha = torch.einsum('eno,no->eno', x_concat, self.att) Dot product instead of matrix multiplication
        alpha = F.leaky_relu(alpha, negative_slope=self.negative_slope)

        # # 2.1.1）: Compute the distance between nodes (using L2 distance as an example)
        # distance = torch.norm(x_i - x_j, p=2, dim=-1)
        # # 2.1.2）: Apply the distance-based penalty to the attention scores
        # alpha = alpha + self.gamma * distance

        # 2.1.3）: Assigning weights to different types of edges
        alpha_weights = torch.ones_like(alpha)  # The default weight is 1
        alpha_weights[edge_type == 0] = 0.1     # Time-dependent edge weight 0.1
        alpha_weights[(edge_type == 1) |(edge_type == 2) | (edge_type == 3) | (edge_type == 4)] = 0.2     # Coupling edge weight 0.2
        alpha_weights[edge_type == 5] = 0.7     # Error correction edge weight 0.7

        alpha_weights_Epsilon = torch.ones_like(alpha)     
        alpha_weights_Epsilon[edge_type == 1] = Epsilon_ij['a2g']     
        alpha_weights_Epsilon[edge_type == 2] = Epsilon_ij['m2g']
        alpha_weights_Epsilon[edge_type == 3] = Epsilon_ij['g2a']     
        alpha_weights_Epsilon[edge_type == 4] = Epsilon_ij['g2m']
           
        # 2.1.4）: Apply edge weights
        alpha = alpha * alpha_weights + alpha_weights_Epsilon

        alpha = softmax(alpha, row)
        alpha = self.attention_dropout(alpha)

        # 2.2）: Multiply attention coefficients with neighbor features
        '''
        Shape:
        x_j: [E, num_heads, out_channels] (Target node features)
        alpha: [E, num_heads]
        alpha.unsqueeze(-1): [E, num_heads, 1]
        message: [E, num_heads, out_channels]
        '''
        message = x_j * alpha.unsqueeze(-1)

        # 2.3）: Aggregate the messages from neighboring nodes
        '''
        Shape:
        aggr_out: [N, num_heads * out_channels] (concat = True)
        aggr_out: [N, out_channels] (concat = False)
        '''
        aggr_out = torch.zeros_like(x) # Initialize aggregation output tensor [N, num_heads, out_channels]
        aggr_out.index_add_(0, row, message) # Accumulate messages for each node

        # Handle output shape based on `concat`
        if self.concat:
            aggr_out = aggr_out.view(-1, self.heads * self.out_channels)  # [N, num_heads * out_channels]
        else:
            aggr_out = aggr_out.mean(dim=1) # [N, out_channels]

        # Return the aggregated features as output
        return aggr_out

In [ ]:
class MGAT_GRU_Classifier(nn.Module):
    def __init__(self, in_channels, hidden_dim, gru_hidden_dim, num_heads, sensor_num, num_class):
        super(MGAT_GRU_Classifier, self).__init__()
        self.sensor_num = sensor_num
        self.num_class = num_class

        self.gat1 = MyGATConv(in_channels, hidden_dim, heads=num_heads)
        self.gru1 = nn.GRU(hidden_dim * num_heads * self.sensor_num, gru_hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(gru_hidden_dim * 2, self.num_class)

    def forward(self, x, edge_index, edge_type):
        x = self.gat1(x, edge_index, edge_type) 
        x = torch.sigmoid(x) * x
        x = x.view(1, -1, x.size(-1) * self.sensor_num) 
        x, _ = self.gru1(x)  
        x = torch.mean(x, dim=1)  
        x = self.fc(x)
        return x 

    

In [ ]:
def train(model, train_data, test_data, batch_size, optimizer, criterion, device):
    model.train()
    total_loss = 0

    train_x = train_data[0]
    train_y = train_data[1]
    num_batches = len(train_x) // batch_size

    for i in range(num_batches):
        batch_loss = 0
        
        optimizer.zero_grad()
        for j in range(batch_size):
            idx = i * batch_size + j

            train_x_hat = train_x[idx]
            node_features, edge_index, edge_type = get_graph(train_x_hat)

            node_features = node_features.to(device)
            edge_index = edge_index.to(device)  
            edge_type = edge_type.to(device)  

            output = model(node_features, edge_index, edge_type)
            label = train_y[idx].to(device) 
            loss = criterion(output.squeeze(0), label)
            batch_loss += loss.item()

            loss.backward()

        optimizer.step()
        avg_batch_loss = batch_loss / batch_size
        total_loss += avg_batch_loss

    test_loss = evaluate_autoencoder(model, test_data)

    avg_loss = total_loss / num_batches
    return avg_loss, test_loss

def evaluate_autoencoder(model, test_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    test_loss = 0

    test_x = test_loader[0]
    test_y = test_loader[1]
    with torch.no_grad():
        for data, label in zip(test_x, test_y):
            test_x_hat = data
            node_features, edge_index, edge_type = get_graph(test_x_hat)

            node_features = node_features.to(device)
            edge_index = edge_index.to(device)  
            edge_type = edge_type.to(device)
             
            output = model(node_features, edge_index, edge_type)
            label = label.to(device)  
            loss = criterion(output.squeeze(0), label)
            test_loss += loss.item()
    test_loss /= test_x.size(0)
    return test_loss

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''
# print(device)
# print(torch.__version__)                # 查看pytorch安装的版本号
# print(torch.cuda.is_available())        # 查看cuda是否可用。True为可用，即是gpu版本pytorch
# print(torch.cuda.get_device_name(0))    # 返回GPU型号
# print(torch.cuda.device_count())        # 返回可以用的cuda（GPU）数量，0代表一个
# print(torch.version.cuda) 
'''
train_data = data_2_model
train_label = label_2_model
print("Train data shape:",train_data.shape)

train_ratio = 0.7
x_train, x_test = train_test_split(train_data, train_size=train_ratio, shuffle=True, random_state=seed)
y_train, y_test = train_test_split(train_label, train_size=train_ratio, shuffle=True, random_state=seed)

train_2_model = [x_train, y_train]
test_2_model = [x_test, y_test]
print(f'x_train_data: {x_train.shape} x_test_data: {x_test.shape}')
print(f'y_train_data: {y_train.shape} y_test_data: {y_test.shape}')


In [ ]:
ReCall = False
if ReCall:
    folder_name = 'FD_MGAT_GRU'
    model_name = 'FD_MGAT_GRU.pth'
    model_path = os.path.join(os.getcwd(), 'info', folder_name, model_name)
    model = torch.load(model_path)
    model.to(device)
else:
    model = MGAT_GRU_Classifier(in_channels=3, hidden_dim=32, gru_hidden_dim=64, num_heads=4, sensor_num = 6, num_class = 4)
    model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epoch_train_loss = np.array([])
epoch_test_loss = np.array([])


In [ ]:
num_epochs = 15
batch_size = 64

for epoch in range(num_epochs):
    train_loss, test_loss = train(model, train_2_model, test_2_model, batch_size, optimizer, criterion, device)
    epoch_train_loss = np.append(epoch_train_loss, train_loss)
    epoch_test_loss = np.append(epoch_test_loss, test_loss)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

In [ ]:
plt.plot(epoch_train_loss)
plt.plot(epoch_test_loss)
plt.xlabel('Epoch')
plt.ylabel('Epoch_Loss')
plt.legend()
plt.show()

In [ ]:
def calculate_accuracy(data, label):
    model.eval()
    acc = 0
    ori_lab = label
    out_lab = np.array([0,0,0,0])
    for i in range(len(data)):
        orign = data[i]
        lab = label[i].numpy()

        node_features, edge_index, edge_type = get_graph(orign)
        node_features = node_features.to(device)
        edge_index = edge_index.to(device)
        edge_type = edge_type.to(device)

        out = model(node_features, edge_index, edge_type).to('cuda')
        out = out.cpu().detach().numpy()
        out_lab = np.vstack((out_lab, out))

        predicted_classes = np.argmax(out)
        if lab[predicted_classes] == 1:
            acc += 1
    accuracy = acc / len(data)
    out_lab = out_lab[1:]
    return accuracy, ori_lab, out_lab

def calculate_one_accuracy(data, label, cnt):
    orign = data[cnt]
    lab = label[cnt].numpy()

    node_features, edge_index, edge_type = get_graph(orign)
    node_features = node_features.to(device)
    edge_index = edge_index.to(device)
    edge_type = edge_type.to(device)

    model.eval()
    out = model(node_features, edge_index, edge_type).to('cuda')
    out = out.cpu().detach().numpy()
    return out, lab

out, lab = calculate_one_accuracy(x_test, y_test, cnt=26)
print(f"out: {out}")
print(f"lab: {lab}")

train_accuracy, ori_train_lab, out_train_lab = calculate_accuracy(x_train, y_train)
test_accuracy, ori_test_lab, out_test_lab = calculate_accuracy(x_test, y_test)
print(f"train_accuracy: {train_accuracy}")
print(f"test_accuracy: {test_accuracy}")

In [ ]:
from sklearn.metrics import confusion_matrix
train_cm = confusion_matrix(np.argmax(ori_train_lab, axis=1), np.argmax(out_train_lab, axis=1))

TP = np.diag(train_cm)
FP = np.sum(train_cm, axis=0) - TP
FN = np.sum(train_cm, axis=1) - TP
TN = np.sum(train_cm) - (TP + FP + FN)

print("Train Dataset:")

print("TP: {}".format(TP))
print("FP: {}".format(FP))
print("FN: {}".format(FN))
print("TN: {}".format(TN))

precision = np.mean(TP / (TP + FP))
recall = np.mean(TP / (TP + FN))
f1_score = 2 * (precision * recall) / (precision + recall)
accuracy = np.sum(TP) / np.sum(train_cm)
print("Precision: {:.4f}  Recall: {:.4f}  F1 Score: {:.4f}  Accuracy: {:.4f}".format(precision, recall, f1_score, accuracy))

test_cm = confusion_matrix(np.argmax(ori_test_lab, axis=1), np.argmax(out_test_lab, axis=1))
print("\nTest Dataset:")
TP = np.diag(test_cm)
FP = np.sum(test_cm, axis=0) - TP
FN = np.sum(test_cm, axis=1) - TP
TN = np.sum(test_cm) - (TP + FP + FN)
print("TP: {}".format(TP))
print("FP: {}".format(FP))
print("FN: {}".format(FN))
print("TN: {}".format(TN))

precision = np.mean(TP / (TP + FP))
recall = np.mean(TP / (TP + FN))
f1_score = 2 * (precision * recall) / (precision + recall)
accuracy = np.sum(TP) / np.sum(test_cm)
print("Precision: {:.4f}  Recall: {:.4f}  F1 Score: {:.4f}  Accuracy: {:.4f}".format(precision, recall, f1_score, accuracy))


In [ ]:
import shutil

ReSaved = True
mode_name = 'FD_MGAT_GRU.pth'
folder_name = mode_name.split('.')[0]
folder_path = os.path.join(os.getcwd(),'info', folder_name)
path = os.path.join(folder_path, mode_name)

if os.path.exists(folder_path):
    if ReSaved:
        shutil.rmtree(folder_path)
        os.makedirs(folder_path, exist_ok=True)
        print(f'Model Re-Saved to {path}')
    else:
        print(f'Model Co-Saved to {path}')
        pass
else:
    os.makedirs(folder_path, exist_ok=True)
    print(f'Model Fi-Saved to {path}')

torch.save(model, path)

train_loss = epoch_train_loss
test_loss = epoch_test_loss
loss_combined = np.column_stack((train_loss, test_loss))
df1 = pd.DataFrame(loss_combined, columns=['train_loss', 'test_loss'])
file_name = os.path.join(folder_path, 'loss.csv')
df1.to_csv(file_name, index=False)
print(f"Successfully saved loss to {file_name}")

accuracy_combined = np.column_stack((train_accuracy, test_accuracy))
df2 = pd.DataFrame(accuracy_combined, columns=['train_accuracy', 'test_accuracy'])
file_name = os.path.join(folder_path, 'accuracy.csv')
df2.to_csv(file_name, index=False)
print(f"Successfully saved accuracy to {file_name}")